In [ ]:
# Movie Recommendation System using Cosine Similarity

In [ ]:
## Step 1: Import Required Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Fix: Use 'latin1' encoding and on_bad_lines='skip' to load the file correctly
movie_data = pd.read_csv('/content/TMDB_movie_dataset_v11.csv', encoding='latin1', on_bad_lines='skip')

# Display the first few rows to verify successful loading
movie_data.head()

In [ ]:
movie_data.columns

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words='english')

# Replace NaN values in 'combined_features' with an empty string
movie_data['combined_features'] = movie_data['combined_features'].fillna('')

# Fit and transform the data to create the TF-IDF matrix
# Using a subset if the dataset is too large, or processing the whole column
tfidf_matrix = tfidf.fit_transform(movie_data['combined_features'].head(20000))

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute the cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Cosine Similarity Matrix Shape: {cosine_sim.shape}")

In [ ]:
# Displaying the title and imdb_id for the first 10 movies in the dataset
movie_data[['title', 'imdb_id']].head(10)

In [ ]:
import pandas as pd

# Calculate the components for the Weighted Rating (IMDb formula)
# W = (v / (v + m)) * R + (m / (v + m)) * C

# v = number of votes for the movie
# m = minimum votes required to be listed
# R = average rating of the movie
# C = mean vote across the whole report

C = movie_data['vote_average'].mean()
m = movie_data['vote_count'].quantile(0.90) # Only consider movies in the top 10% of vote counts

def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m) * R) + (m/(m+v) * C)

# Apply the formula to a copy of the dataframe to avoid settings warnings
q_movies = movie_data.copy().loc[movie_data['vote_count'] >= m]
q_movies['weighted_score'] = q_movies.apply(weighted_rating, axis=1)

# Sort and display the top 10 movies by weighted rating
top_rated = q_movies[['title', 'vote_count', 'vote_average', 'weighted_score', 'imdb_id']].sort_values('weighted_score', ascending=False)

print(f"Mean Rating (C): {C:.2f}")
print(f"Minimum Votes Required (m): {m}")
display(top_rated.head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare data for visualization from the top_rated DataFrame
viz_data = top_rated.head(10)

plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

# Create the bar plot - updated syntax to avoid FutureWarnings
ax = sns.barplot(
    x='weighted_score',
    y='title',
    data=viz_data,
    hue='title',
    palette='viridis',
    legend=False
)

# Add labels and title
plt.xlabel('Weighted Score', fontsize=12)
plt.ylabel('Movie Title', fontsize=12)
plt.title('Top 10 Movies by Weighted Rating Score', fontsize=15)

# Adjust x-axis to show scores more clearly
plt.xlim(viz_data['weighted_score'].min() - 0.5, viz_data['weighted_score'].max() + 0.1)

plt.tight_layout()
plt.show()

In [ ]:
# 1. Extract all unique genre words across the dataset
all_genres = movie_data['genres'].fillna('').str.split(', ')
unique_genres = sorted(list(set([genre for sublist in all_genres for genre in sublist if genre])))

print(f'Total Unique Genres Found: {len(unique_genres)}')
print('Unique Genre List:', unique_genres)

# 2. Display every genre related to each movie (first 10 for preview)
print('\nGenre breakdown per movie:')
display(movie_data[['title', 'genres']].head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Formatting the Summary Table (Styled for clarity)
# We limit to top 15 genres for the visual, but keep all for the table if needed
top_genres = genre_summary.head(15)

def highlight_top(s):
    return ['background-color: #e8f5e9' if v > 10 else '' for v in s]

print("📊 Genre Distribution Summary")
display(genre_summary.style
        .format({'Percentage': '{:.2f}%'})
        .bar(subset=['Percentage'], color='#5fba7d')
        .set_caption("Full Genre Breakdown")
        .hide(axis='index'))

# 2. Visualizing with a Chart
plt.figure(figsize=(12, 8))
sns.set_style("whitegrid")

# Create the bar plot
ax = sns.barplot(
    x='Count',
    y='Genre',
    data=top_genres,
    palette='viridis',
    hue='Genre',
    legend=False
)

# Add percentage labels to the end of each bar
for i, p in enumerate(ax.patches):
    percentage = top_genres['Percentage'].iloc[i]
    ax.annotate(f'{percentage:.1f}%',
                (p.get_width() + 5, p.get_y() + p.get_height() / 2),
                va='center', fontsize=10, fontweight='bold')

plt.title('Top 15 Movie Genres by Frequency', fontsize=16, pad=20)
plt.xlabel('Number of Movies', fontsize=12)
plt.ylabel('Genre', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import HTML

# Create a styled HTML table
styled_html = (
    genre_df.style
    .background_gradient(cmap='Greens', subset=['Count'])
    .format({'Percentage': '{:.2f}%'})
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#1a237e'), ('color', 'white'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'left'), ('padding', '10px')]}
    ])
    .hide(axis='index')
    .to_html()
)

print("📋 Genre Analytics Report")
display(HTML(styled_html))

In [ ]:
import pandas as pd

# 1. Prepare the data for genre-based grouping
df_genres = movie_data.copy()
df_genres['genre_list'] = df_genres['genres'].fillna('').str.split(', ')
df_exploded = df_genres.explode('genre_list')

# 2. Use the IMDb weighted rating formula calculated earlier
C = df_genres['vote_average'].mean()
m = df_genres['vote_count'].quantile(0.90)

def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m) * R) + (m/(m+v) * C)

df_exploded['score'] = df_exploded.apply(weighted_rating, axis=1)

# 3. Define unique genres (filtering out empty strings)
unique_genres = [g for g in sorted(df_exploded['genre_list'].unique()) if g]

# 4. Display top 5 movies for each genre in a clean table
print("Top 5 Movies per Genre (Weighted Rating)")
for genre in unique_genres:
    genre_subset = df_exploded[df_exploded['genre_list'] == genre]\
                   .sort_values('score', ascending=False)\
                   .head(5)

    if not genre_subset.empty:
        print(f"\nGenre: {genre}")
        display(genre_subset[['title', 'vote_average', 'vote_count', 'score']].reset_index(drop=True))

In [ ]:
import pandas as pd

# 1. Direct approach: use the original dataframe to ensure we don't miss data from previous filters
df_rank = movie_data.copy()

# Ensure genres are string and split them
df_rank['genre_split'] = df_rank['genres'].fillna('').str.split(', ')
df_exploded = df_rank.explode('genre_split')

# Recalculate components for this specific view to ensure results
C_global = df_rank['vote_average'].mean()
m_local = df_rank['vote_count'].quantile(0.85) # slightly more inclusive top 15%

def calculate_weighted(x, m=m_local, C=C_global):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m) * R) + (m/(m+v) * C)

df_exploded['weighted_score_local'] = df_exploded.apply(calculate_weighted, axis=1)

genres_to_show = ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']

print(f"Top 5 Movies by Weighted Rating per Genre (Min Votes: {m_local}):")
for genre in genres_to_show:
    # Case-insensitive match and strip whitespace, then sort by weighted_score_local descending
    genre_top = df_exploded[df_exploded['genre_split'].str.strip().str.lower() == genre.lower()]\
                .sort_values('weighted_score_local', ascending=False)\
                .head(5)

    if not genre_top.empty:
        print(f"\n--- {genre} ---")
        display(genre_top[['title', 'weighted_score_local', 'vote_count', 'vote_average']])
    else:
        print(f"\n--- {genre} (No high-scoring movies found) ---")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd # Ensure pandas is explicitly imported

# 1. Ensure the Score is calculated
C_glob = movie_data['vote_average'].mean()
m_loc = movie_data['vote_count'].quantile(0.85)

def weighted_rating_func(x, m=m_loc, C=C_glob):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m) * R) + (m/(m+v) * C)

movie_data['weighted_score_local'] = movie_data.apply(weighted_rating_func, axis=1)

# 2. Setup Search Widgets
unique_titles = sorted(movie_data['title'].astype(str).unique().tolist())

movie_input = widgets.Combobox(
    placeholder='Search movie (e.g. Inception)',
    options=unique_titles,
    description='🎬 Search:',
    ensure_option=True,
    layout=widgets.Layout(width='350px'),
    style={'description_width': 'initial'}
)

recommend_btn = widgets.Button(
    description='Get Recommendations',
    button_style='success',
    icon='magic'
)
rec_output = widgets.Output()

# 3. The Visual Recommendation Logic
def on_button_clicked(b):
    with rec_output:
        clear_output()
        title = movie_input.value.strip()

        if not title:
            print("⚠️ Please enter a movie title.")
            return

        # 1. Get raw recommendations
        results = get_recommendations(title)

        if isinstance(results, str):
            print(f"❌ {results}")
            return

        # 2. Ensure results is a DataFrame
        if isinstance(results, (list, pd.Series)):
            results = pd.DataFrame(results, columns=['title'])

        # 3. Merge with ALL necessary columns
        showcase_cols = ['title', 'weighted_score_local', 'vote_average', 'vote_count', 'budget', 'revenue', 'release_date']
        existing_cols = [col for col in showcase_cols if col in movie_data.columns]

        final_rec = results.merge(movie_data[existing_cols], on='title', how='left')

        # 4. Cleanup
        final_rec = final_rec.dropna(subset=['title']).drop_duplicates(subset=['title'])
        if 'weighted_score_local' in final_rec.columns:
            final_rec = final_rec.sort_values('weighted_score_local', ascending=False)

        # 5. Styling
        available_for_gradient = [c for c in ['weighted_score_local', 'vote_average'] if c in final_rec.columns]

        styled_rec = final_rec.style.set_properties(**{
            'background-color': '#ffffff',
            'color': '#333',
            'border': '1px solid #eee'
        })

        if available_for_gradient:
            styled_rec = styled_rec.background_gradient(subset=available_for_gradient, cmap='RdYlGn')

        # Formatting
        format_dict = {}
        if 'weighted_score_local' in final_rec.columns: format_dict['weighted_score_local'] = '{:.2f}'
        if 'vote_average' in final_rec.columns: format_dict['vote_average'] = '⭐ {:.1f}'
        if 'budget' in final_rec.columns: format_dict['budget'] = '${:,.0f}'
        if 'revenue' in final_rec.columns: format_dict['revenue'] = '${:,.0f}'

        styled_rec = styled_rec.format(format_dict).hide(axis='index')

        display(styled_rec)

# --- CRITICAL MISSING LINE ---
recommend_btn.on_click(on_button_clicked)
# -----------------------------

# Layout Display
display(widgets.VBox([
    widgets.HTML("<h2 style='color: #1a237e;'>🤖 AI Movie Recommender</h2>"),
    widgets.HBox([movie_input, recommend_btn]),
    rec_output
]))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Reuse the exploded genre dataframe
df_rank = movie_data.copy()
df_rank['genre_split'] = df_rank['genres'].fillna('').str.split(', ')
df_exploded = df_rank.explode('genre_split')

# 2. Setup the selection widget
unique_genres = sorted([g.strip() for g in df_exploded['genre_split'].unique() if g])
genre_dropdown = widgets.Dropdown(
    options=unique_genres,
    value=unique_genres[0] if unique_genres else None,
    description='🎯 Genre:',
    style={'description_width': 'initial'}
)

button = widgets.Button(
    description="Analyze Genre",
    button_style='info', # Blue themed
    icon='search'
)
output_area = widgets.Output()

def on_genre_button_clicked(b):
    with output_area:
        clear_output()
        selected_genre = genre_dropdown.value

        # Filter and sort
        results = df_exploded[df_exploded['genre_split'].str.strip() == selected_genre]\
                  .sort_values('weighted_score_local', ascending=False).head(10)

        if results.empty:
            print(f"No movies found for {selected_genre}.")
            return

        # Prepare a clean version for display
        display_df = results[['title', 'weighted_score_local', 'vote_average', 'vote_count', 'release_date']].copy()
        display_df.columns = ['Movie Title', 'Score', 'Avg Rating', 'Total Votes', 'Released']

        # 3. Enhanced Visualization using Pandas Styler
        styled_table = display_df.style.set_properties(**{
            'background-color': '#f9f9f9',
            'color': 'black',
            'border-color': 'white'
        }).background_gradient(
            subset=['Score', 'Avg Rating'],
            cmap='YlGn' # Yellow to Green gradient
        ).format({
            'Score': '{:.2f}',
            'Avg Rating': '⭐ {:.1f}',
            'Total Votes': '{:,}'
        }).set_table_styles([
            {'selector': 'th', 'props': [('font-size', '12pt'), ('background-color', '#2e7d32'), ('color', 'white')]}
        ]).hide(axis='index')

        print(f"🏆 TOP 10 {selected_genre.upper()} MOVIES\n")
        display(styled_table)

button.on_click(on_genre_button_clicked)

# Layout
display(widgets.VBox([
    widgets.HTML("<h2>Genre Insights Dashboard</h2>"),
    widgets.HBox([genre_dropdown, button]),
    output_area
]))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Prepare the movie selection list
all_titles = sorted(movie_data['title'].astype(str).unique().tolist())

# 2. Create the widgets
movie_selector = widgets.Combobox(
    placeholder='Type Movie Name (e.g. Inception)',
    options=all_titles, # Keep the full list here for built-in filtering
    description='Movie Name:',
    ensure_option=True,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

get_details_btn = widgets.Button(
    description='Show Movie Details',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

details_output = widgets.Output()

# 3. The Logic Function
def display_movie_details(b):
    with details_output:
        clear_output()
        # Strip whitespace to prevent "Title " vs "Title" errors
        selected_title = movie_selector.value.strip()

        if not selected_title:
            print("⚠️ Please type or select a movie name first.")
            return

        # Filter the dataset - Using case-insensitive matching for safety
        movie_match = movie_data[movie_data['title'].str.lower() == selected_title.lower()]

        if not movie_match.empty:
            info = movie_match.iloc[0]

            # Formatting the display
            print(f"\n{'='*60}")
            print(f"🎬 MOVIE DETAILS: {info['title'].upper()}")
            print(f"{'='*60}\n")

            # Using .get() with defaults to avoid KeyErrors
            print(f"{'Tagline:':<15} {info.get('tagline', 'N/A')}")
            print(f"{'Genres:':<15} {info.get('genres', 'N/A')}")
            print(f"{'Release Date:':<15} {info.get('release_date', 'N/A')}")
            print(f"{'Runtime:':<15} {info.get('runtime', 0)} mins")
            print(f"{'Rating:':<15} {info.get('vote_average', 0)} ({info.get('vote_count', 0)} votes)")

            # Use 0 as default if weighted_score doesn't exist yet
            ws = info.get('weighted_score_local', 0)
            print(f"{'Weighted Scr:':<15} {ws:.2f}" if isinstance(ws, (int, float)) else f"{'Weighted Scr:':<15} N/A")

            print(f"{'Budget:':<15} ${info.get('budget', 0):,}")
            print(f"{'Revenue:':<15} ${info.get('revenue', 0):,}")
            print(f"{'Status:':<15} {info.get('status', 'N/A')}")

            print(f"\n{'OVERVIEW:':<15}\n{info.get('overview', 'No description available.')}")
            print(f"\n{'='*60}")
        else:
            print(f"❌ Could not find '{selected_title}'.")
            print("Tip: Start typing and select an option from the dropdown to ensure a match.")

get_details_btn.on_click(display_movie_details)

# 4. Layout display
display(widgets.VBox([
    widgets.HTML("<h2 style='color: #2e7d32;'>🎥 Movie Metadata Explorer</h2>"),
    widgets.HBox([movie_selector, get_details_btn]),
    details_output
]))

In [ ]:
from IPython.display import clear_output
clear_output()